In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os

# 获取项目根目录：当前笔记本文件的目录再上一级
project_root = os.path.abspath('..')   # 如果笔记本在 notebook/ 下，否则改为 os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f"项目根目录: {project_root}")

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [ ]:
# 写一个快速排序函数

In [ ]:
#load_longbench_dataset_with_chunking()

# load_longbench_with_refined_chunking()


#chunks = split_chunks_with_cleaning(file_path)

#print(f"清洗并分块完成，共提取出 {len(chunks)} 条高质量文档块。")

# 预览一下清洗后的效果
# for i, chunk in enumerate(chunks[:3], 1):
#     print(f"--- 经清洗后的块 {i} ---")
#     print(f"{chunk[:200]}") # 看看是否还有 <br>
#     print()

#update_eval_file('longbench_eval.json', 'longbench_eval_with_gt.json')

# --- 执行示例 ---
# 1. 运行你之前的 generate_contextual_chunks 得到列表
#enhanced_chunks = generate_contextual_chunks('longbench_kb.md')

# 2. 保存为原样格式的文件
#save_enhanced_kb_to_disk(enhanced_chunks, "longbench_kb_contextual.md")

#save_embeddings(real_ids, chunks, embeddings)
#print(real_ids[0],embeddings[0])
#client.delete_collection("baai_bge_collection")
#client.delete_collection("contextual_collection")

In [ ]:

#real_ids, chunks = parse_kb_file('dureader_kb.md')
#embeddings = embedding_model.encode(chunks, normalize_embeddings=True).tolist()
#save_embeddings(real_ids, chunks, embeddings)

In [ ]:
import sys
sys.path.append('/data1/yesiliang/rag/')
import pickle
import numpy as np
from src.rerank_utils import colbert_model, parse_kb_file  # 确保 colbert_model 已加载

# ---------- 配置 ----------
KB_FILE = "/data1/yesiliang/rag/data/dureader/dureader_kb.md"          # 新知识库文件
CACHE_FILE = "/data1/yesiliang/rag/vector_db/colbert_cache_dureader.pkl"

# ---------- 1. 解析知识库 ----------
print(f"🔍 解析知识库: {KB_FILE}")
real_ids, real_chunks = parse_kb_file(KB_FILE)
print(f"   共解析出 {len(real_ids)} 个文本块")

# ---------- 2. 批量计算 ColBERT 多向量 ----------
print(f"🚀 开始批量计算 ColBERT 向量（共 {len(real_chunks)} 个块）...")
outputs = colbert_model.encode(real_chunks, return_colbert_vecs=True)
colbert_vecs = outputs['colbert_vecs']  # list of numpy arrays, 每个 [Ld, D]

# ---------- 3. 建立 ID -> 向量的映射 ----------
print("📦 建立 ID -> 向量缓存...")
cache = {}
for chunk_id, vec in zip(real_ids, colbert_vecs):
    cache[chunk_id] = np.asarray(vec)  # 转成 numpy 数组，方便后续使用

# ---------- 4. 保存到磁盘 ----------
with open(CACHE_FILE, 'wb') as f:
    pickle.dump(cache, f)

print(f"✅ 预计算完成！缓存已保存至: {CACHE_FILE} (共 {len(cache)} 个向量)")

In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
sys.path.append('/data1/yesiliang/rag/')
# 小测试：验证问题拆解函数
from src.rerank_utils import decompose_question

test_question = "南通海安是不是很穷"

print(f"原始问题: {test_question}\n")
subs = decompose_question(test_question)
print(f"拆解出 {len(subs)} 个子问题:")
for i, sq in enumerate(subs, 1):
    print(f"  {i}. {sq}")

In [ ]:
import json
import time
import sys
import torch
import gc
sys.path.append('/data1/yesiliang/rag/')
from src.rerank_utils import get_chunks_by_question_id, decompose_question, bge_pointwise_rerank, generate

# 我只对前 3 个问题做循环，不打答案，只看显存
with open("dureader_eval.json", "r", encoding="utf-8") as f:
    test_list = json.load(f)["test"][:3]

def print_gpu_mem(step_label):
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1024**3
        print(f"   [{step_label}] 显存已分配: {mem:.2f} GB")

for idx, qa in enumerate(test_list):
    q = qa["question"]
    print(f"\n==== 处理第 {idx} 题 ====")
    print_gpu_mem("BEFORE chunk load")   # 查看当前显存

    chunks = get_chunks_by_question_id(idx)
    print_gpu_mem("AFTER chunk load")   # 获取题目文本块后的显存

    subs = decompose_question(q)        # 调用你的拆解函数
    print_gpu_mem("AFTER decompose")    # 拆解后的显存

    ids = [f"题_{idx}_块_{j}" for j in range(len(chunks))]
    for _k, sub_q in enumerate(subs):
        _ = bge_pointwise_rerank(sub_q, ids, chunks)
    print_gpu_mem("AFTER bge rerank")   # 进行重排后的显存

    # 没用 del 和 empty_cache 时，显存理论上会一点点累积上去
    # 如果这里打印出的数字一直上升，就是 OOM 了。

    time.sleep(1)  # 稍微喘口气，方便你看日志

In [ ]:
from src.generate import (
    evaluate_rag_system,
    strategy_pairwise,
    strategy_listwise,
    strategy_colbert,
    strategy_pointwise,
    strategy_baseline,
    strategy_colbert_listwise,
    strategy_setselection,
    # strategy_listwise_setselection,
    # strategy_pointwise_dureader
)
import json

# 1. 加载数据
with open("/data1/yesiliang/rag/data/mutifieldqa/mutifieldqa_eval.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)
test_questions = test_data["test"]

# 2. 【核心修改点】在这里选择你想使用的策略函数名
#CURRENT_STRATEGY = strategy_baseline
#CURRENT_STRATEGY = strategy_pointwise
CURRENT_STRATEGY = strategy_listwise
#CURRENT_STRATEGY = strategy_pairwise
#CURRENT_STRATEGY = strategy_colbert 
#CURRENT_STRATEGY = strategy_colbert_listwise 
#CURRENT_STRATEGY = strategy_setselection
#CURRENT_STRATEGY = strategy_listwise_setselection
#CURRENT_STRATEGY = strategy_pointwise_dureader
# 3. 设置对应的输出文件名
output_mapping = {
    strategy_baseline: "multifieldqa_baseline.json",
    strategy_pointwise: "multifieldqa_pointwise.json",
    strategy_colbert: "multifieldqa_colbert.json",
    strategy_listwise: "multifieldqa_listwise.json",
    strategy_pairwise: "multifieldqa_pairwise.json",
    strategy_colbert_listwise: "bge_colbert_deepseek_listwise.json",
    strategy_setselection:"dureader_setselection.json",
    # strategy_listwise_setselection: "dureader_listwise_setselection.json",
    # strategy_pointwise_dureader: "dureader_pointwise.json"
}

output_file = output_mapping[CURRENT_STRATEGY]
output_path = f"/data1/yesiliang/rag/results/{output_file}" 
# 4. 执行并保存
test_results = evaluate_rag_system(test_questions, CURRENT_STRATEGY, top_k_for_gen=2, skip_retrieval=False)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(test_results, f, indent=2, ensure_ascii=False)

print(f"评估完成！使用策略: {CURRENT_STRATEGY.__name__}, 结果存至: {output_path}")

In [ ]:
from src.generate import calculate_metrics

GT_FILE = '/data1/yesiliang/rag/data/mutifieldqa/mutifieldqa_eval.json'
PRED_FILE = '/data1/yesiliang/rag/results/multifieldqa_listwise.json' 
    
calculate_metrics(GT_FILE, PRED_FILE)

In [ ]:
from src.generate import run_gen_evaluation

run_gen_evaluation("/data1/yesiliang/rag/results/multifieldqa_listwise.json")

In [ ]:
from pathlib import Path
import json


def trim_results_to_gold(source_dir: str = "/data1/yesiliang/rag/results", target_dir: str = "/data1/yesiliang/rag/results_trimmed"):
    source_dir = Path(source_dir)
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    def build_trimmed_item(item: dict):
        if not isinstance(item, dict):
            return None

        gold_answer = (
            item.get("gold_answer")
            or item.get("reference_answer")
            or item.get("standard_answer")
            or item.get("answer")
            or item.get("ground_truth")
            or item.get("label")
            or ""
        )
        generated_answer = (
            item.get("generated_answer")
            or item.get("generated_answer")
            or item.get("prediction")
            or item.get("answer")
            or ""
        )

        return {
            "question": item.get("question", ""),
            "gold_answer": gold_answer,
            "generated_answer": generated_answer,
        }

    for path in sorted(source_dir.glob("*.json")):
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            trimmed = [build_trimmed_item(item) for item in data if isinstance(item, dict)]
        elif isinstance(data, dict):
            trimmed = {
                key: build_trimmed_item(item)
                for key, item in data.items()
                if isinstance(item, dict)
            }
        else:
            raise ValueError(f"Unsupported JSON structure in {path}")

        out_path = target_dir / path.name
        with out_path.open("w", encoding="utf-8") as f:
            json.dump(trimmed, f, ensure_ascii=False, indent=2)

        if isinstance(trimmed, list):
            count = len(trimmed)
        else:
            count = len(trimmed)
        print(f"已覆盖 {out_path}，共处理 {count} 条记录")


# 小测试：只处理一个 source_dir 下的文件
trim_results_to_gold()


In [ ]:
import os
import json
import re
from pathlib import Path
from time import sleep
from dotenv import load_dotenv
from openai import OpenAI

# 加载环境变量
load_dotenv("/data1/yesiliang/rag/config/.env")

deepseek_api_key = os.environ.get("DEEPSEEK_API_KEY")
if not deepseek_api_key:
    raise ValueError("请先在 /data1/yesiliang/rag/config/.env 中设置 DEEPSEEK_API_KEY")

deepseek_client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com"
)

def evaluate_answers(input_file_path: str, output_file_path: str, sample_size: int | None = None, show_progress: bool = True):
    input_file = Path(input_file_path)
    output_file = Path(output_file_path)

    with input_file.open("r", encoding="utf-8") as f:
        records = json.load(f)

    if sample_size is not None:
        records = records[:sample_size]
        print(f"小测试模式：只处理前 {sample_size} 条记录")

    prompt_template = """### 角色扮演
    你将扮演一个公正、严格的评估专家。你的任务是根据[用户问题]和[标准参考答案]，评估[生成答案]的质量。

    ### 评估依据
    - [用户问题]: {question}
    - [标准参考答案]: {reference}
    - [生成答案]: {answer}

    ### 评价维度
    请基于以下三个维度，将[生成答案]与[标准参考答案]进行对比，每个维度的分数为1（极差）到5（优秀）：
    1. **语义等价性 (Semantic Equivalence)**: [生成答案]的核心含义、事实陈述与[标准参考答案]是否一致？是否存在遗漏、歪曲或相反的信息？
    2. **关键信息覆盖 (Key Information Coverage)**: [标准参考答案]中明确包含的关键实体、数值、日期或限定条件，在[生成答案]中是否被准确、完整地覆盖？
    3. **无关冗余 (Irrelevant Redundancy)**: [生成答案]是否包含与[标准参考答案]和[用户问题]无关的扩展信息或废话？注意：额外信息不一定扣分，但如果这些信息淡化了核心答案或引入了模糊性，则应低分。

    ### 输出格式
    请严格以JSON格式输出你的评估结果，并为每个评分附上简洁的理由：
    {{
      "semantic_equivalence": {{ "score": Integer, "reason": "string" }},
      "key_info_coverage": {{ "score": Integer, "reason": "string" }},
      "irrelevant_redundancy": {{ "score": Integer, "reason": "string" }}
    }}
    """

    def parse_json_response(text: str):
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            match = re.search(r"(\{(?:.|\n)*\})", text)
            if match:
                try:
                    return json.loads(match.group(1))
                except json.JSONDecodeError:
                    pass
        return {"model_output": text.strip()}

    evaluations = []
    total = len(records)
    for idx, item in enumerate(records, start=1):
        question = item.get("question", "")
        answer = item.get("generated_answer", "")
        reference = (
            item.get("gold_answer")
            or item.get("reference_answer")
            or item.get("standard_answer")
            or item.get("answer")
            or item.get("ground_truth")
            or ""
        )
        prompt = prompt_template.format(question=question, reference=reference, answer=answer)

        response = deepseek_client.chat.completions.create(
            model="deepseek-v4-pro",
            messages=[
                {"role": "system", "content": "你是一位公正、严格的评估专家。"},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=512
        )

        content = response.choices[0].message.content
        parsed = parse_json_response(content)

        evaluations.append({
            "question": question,
            "gold_answer": reference,
            "generated_answer": answer,
            "evaluation": parsed,
            "raw_model_output": content,
        })

        if show_progress:
            print(f"[{idx}/{total}] 已处理：{question[:60]}")

        sleep(0.8)

    with output_file.open("w", encoding="utf-8") as f:
        json.dump(evaluations, f, ensure_ascii=False, indent=2)

    print(f"已保存评估结果到: {output_file}")
    return evaluations


def evaluate_all_tests(test_map: dict, sample_size: int | None = None):
    summary = {}
    total_files = len(test_map)
    for idx, (input_path, output_path) in enumerate(test_map.items(), start=1):
        print(f"\n=== 运行文件 {idx}/{total_files}: {Path(input_path).name} ===")
        evaluate_answers(input_path, output_path, sample_size=sample_size)
        summary[Path(input_path).name] = output_path
    print("\n=== 一键测试完成 ===")
    return summary

#小测试示例：取消注释并修改为你想测试的文件
#evaluate_answers("/data1/yesiliang/rag/results_trimmed/multifieldqa_baseline.json", "/data1/yesiliang/rag/results_trimmed/multifieldqa_baseline_deepseek_eval.json", sample_size=3)

#一键运行示例：取消注释并运行所有文件
ALL_TESTS = {
    "/data1/yesiliang/rag/results_trimmed/colbert_listwise.json": "/data1/yesiliang/rag/results_trimmed/colbert_listwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_baseline.json": "/data1/yesiliang/rag/results_trimmed/dureader_baseline_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_colbert.json": "/data1/yesiliang/rag/results_trimmed/dureader_colbert_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_decomposed_pointwise.json": "/data1/yesiliang/rag/results_trimmed/dureader_decomposed_pointwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_listwise_setselection.json": "/data1/yesiliang/rag/results_trimmed/dureader_listwise_setselection_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_pointwise.json": "/data1/yesiliang/rag/results_trimmed/dureader_pointwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_setselection.json": "/data1/yesiliang/rag/results_trimmed/dureader_setselection_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/listwise_contextual.json": "/data1/yesiliang/rag/results_trimmed/listwise_contextual_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/multifieldqa_baseline.json": "/data1/yesiliang/rag/results_trimmed/multifieldqa_baseline_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/multifieldqa_colbert.json": "/data1/yesiliang/rag/results_trimmed/multifieldqa_colbert_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/multifieldqa_listwise.json": "/data1/yesiliang/rag/results_trimmed/multifieldqa_listwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/multifieldqa_pairwise.json": "/data1/yesiliang/rag/results_trimmed/multifieldqa_pairwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/multifieldqa_pointwise.json": "/data1/yesiliang/rag/results_trimmed/multifieldqa_pointwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/pointwise_contextual.json": "/data1/yesiliang/rag/results_trimmed/pointwise_contextual_deepseek_eval.json",
}
evaluate_all_tests(ALL_TESTS)


In [ ]:
from pathlib import Path
import os
import json
import re
from pathlib import Path
from time import sleep
def calculate_average_scores(eval_file_path: str, num_questions: int = 50):
    eval_file = Path(eval_file_path)

    with eval_file.open("r", encoding="utf-8") as f:
        evaluations = json.load(f)

    evaluations = evaluations[:num_questions]
    semantic_scores = []
    key_info_scores = []
    redundancy_scores = []

    def extract_score(metric):
        if isinstance(metric, dict):
            score = metric.get("score")
            if score is None:
                score = metric.get("mean score")
            try:
                return float(score)
            except (TypeError, ValueError):
                return None
        return None

    for item in evaluations:
        eval_data = item.get("evaluation", {})
        semantic_scores.append(extract_score(eval_data.get("semantic_equivalence", {})))
        key_info_scores.append(extract_score(eval_data.get("key_info_coverage", {})))
        redundancy_scores.append(extract_score(eval_data.get("irrelevant_redundancy", {})))

    def avg(scores):
        valid = [s for s in scores if s is not None]
        return sum(valid) / len(valid) if valid else None

    avg_semantic = avg(semantic_scores)
    avg_key_info = avg(key_info_scores)
    avg_redundancy = avg(redundancy_scores)

    print(f"\n=== 平均分结果（前 {num_questions} 道题） ===")
    if avg_semantic is not None:
        print(f"语义等价性 (Semantic Equivalence): {avg_semantic:.2f}")
    else:
        print("语义等价性 (Semantic Equivalence): 无有效评分")

    if avg_key_info is not None:
        print(f"关键信息覆盖 (Key Information Coverage): {avg_key_info:.2f}")
    else:
        print("关键信息覆盖 (Key Information Coverage): 无有效评分")

    if avg_redundancy is not None:
        print(f"无关冗余 (Irrelevant Redundancy): {avg_redundancy:.2f}")
    else:
        print("无关冗余 (Irrelevant Redundancy): 无有效评分")

    return {
        "semantic_equivalence": avg_semantic,
        "key_info_coverage": avg_key_info,
        "irrelevant_redundancy": avg_redundancy,
    }


def calculate_all_average_scores(eval_file_paths: list[str], num_questions: int = 50):
    results = {}
    total_files = len(eval_file_paths)
    for idx, path in enumerate(eval_file_paths, start=1):
        print(f"\n[{idx}/{total_files}] 计算文件: {Path(path).name}")
        results[path] = calculate_average_scores(path, num_questions=num_questions)
    print("\n=== 所有文件平均分计算完成 ===")
    return results

 #小测试示例：取消注释并修改路径
#calculate_average_scores("/data1/yesiliang/rag/results_trimmed/multifieldqa_baseline_deepseek_eval.json", 5)

#一键计算示例：取消注释并修改路径列表
calculate_all_average_scores([
    "/data1/yesiliang/rag/results_trimmed/multifieldqa_baseline_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/multifieldqa_colbert_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/multifieldqa_listwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/multifieldqa_pointwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/multifieldqa_pairwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/pointwise_contextual_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/colbert_listwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/listwise_contextual_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_pointwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_baseline_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_decomposed_pointwise_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_colbert_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_listwise_setselection_deepseek_eval.json",
    "/data1/yesiliang/rag/results_trimmed/dureader_setselection_deepseek_eval.json"
], num_questions=50)
